# S0: Temporal Persistence Profiling

**Goal**: Measure oracle δ = ‖h(t) - h(t-1)‖ / ‖h(t-1)‖ (ground truth channel variation)

**Key questions**:
1. How much does the channel change between consecutive slots? (go/no-go)
2. Static δ ≈ 0? (simulator noise floor sanity check)  
3. NMSE if we reuse previous estimate?

Run cells sequentially. Data loads once and stays in memory.

In [1]:
import sys; sys.path.insert(0, "../../..")  # project root
from src.experiments.S0_persistence.core import load_data, run_all_bs, check_go_nogo, save_results, load_results
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-whitegrid")

PRESET = "munich_elaa_m_1k_15g"  # has h5 data ready
MAX_SNAPSHOTS = 20000

In [2]:
# Load data (slow — stays in memory for re-analysis)
# Or load cached results if available
GPU = "0,1,2,3,4,5,6"             # GPU(s) to use
UE_PER_SPEED = 2      # 2 UEs per speed category (static/ped/veh) = 6 total
SNAP_LIMIT = 4000     # snapshots per UE

result = load_results(PRESET)
if result is None:
    print("No cached results, computing...")
    data = load_data(PRESET, max_snapshots=MAX_SNAPSHOTS)
    result = run_all_bs(data, PRESET, gpu=GPU, max_snapshots=SNAP_LIMIT, ue_per_speed=UE_PER_SPEED)
    save_results(result)
    print("Saved results. Data stays in memory.")
else:
    print(f"Loaded cached results: {result['summary']['n_ues']} UEs")

per_ue = result["per_ue"]
summary = result["summary"]
print(f"\nSummary: median δ={summary['median_delta']:.4f}, "
      f"static δ={summary['static_median_delta']:.4f}, "
      f"NMSE(reuse)={summary['median_nmse_reuse_db']:.1f}dB")
print(f"\nPer UE:")
for u in per_ue:
    label = 'static' if u['speed']<0.1 else ('ped' if u['speed']<5 else 'veh')
    print(f"  UE {u['uid']} ({label}): δ={u['median_delta']:.4f}, NMSE={u['nmse_reuse_db']:.1f}dB")

No cached results, computing...
Loaded munich_elaa_m_1k_15g: 10000 snapshots, 9 UEs, dt=0.00025s
  BS 0: 0 UEs, skipping
  BS 2: 0 UEs, skipping
  BS 3: 0 UEs, skipping


CIR→CFR (7 GPU):   0%|          | 0/3 [00:00<?, ?chunk/s]

  BS 1: 5 UEs, median δ=0.0074, static δ=0.0043
Saved: assets/results/S0_persistence/persistence_munich_elaa_m_1k_15g.json
Saved results. Data stays in memory.

Summary: median δ=0.0074, static δ=0.0043, NMSE(reuse)=-20.1dB

Per UE:
  UE 0 (static): δ=0.0074, NMSE=-20.1dB
  UE 1 (static): δ=0.0013, NMSE=-20.3dB
  UE 3 (ped): δ=0.0000, NMSE=-20.4dB
  UE 4 (ped): δ=0.0427, NMSE=-19.4dB
  UE 6 (veh): δ=0.1983, NMSE=-12.3dB


In [ ]:
# Plot: Per-UE results — Oracle δ vs LS δ
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

speeds = np.array([u["speed"] for u in per_ue])
deltas_oracle = np.array([u["median_delta"] for u in per_ue])
deltas_ls = np.array([u.get("median_delta_ls", u["median_delta"]) for u in per_ue])
nmses = np.array([u["nmse_reuse_db"] for u in per_ue])

# 1a: Speed vs δ — Oracle AND LS side by side
ax = axes[0]
for i, u in enumerate(per_ue):
    spd = u["speed"]
    d_o = u["median_delta"]
    d_l = u.get("median_delta_ls", d_o)
    ax.scatter(spd - 0.15, d_o, c="tab:blue", s=80, edgecolors="k", linewidths=0.5, zorder=5,
               marker="o", label="Oracle δ" if i == 0 else "")
    ax.scatter(spd + 0.15, d_l, c="tab:orange", s=80, edgecolors="k", linewidths=0.5, zorder=5,
               marker="s", label="LS δ (20dB)" if i == 0 else "")
    ax.annotate(f"UE{u['uid']}", (spd + 0.2, max(d_o, d_l)),
                textcoords="offset points", xytext=(5, 3), fontsize=7)
ax.axhline(summary["static_median_delta"], color="red", ls="--", alpha=0.5,
           label=f"static noise floor ({summary['static_median_delta']:.4f})")
ax.set_xlabel("Speed (m/s)")
ax.set_ylabel("Median δ")
ax.set_title("Speed → Channel Variation (Oracle vs LS)")
ax.set_xlim(-0.5, 9.5)
ax.legend(fontsize=7, loc="upper left")

# 1b: Grouped bar — Oracle vs LS by speed category
ax = axes[1]
cats = {"static": 0.0, "ped\n(1m/s)": 1.0, "veh\n(8.3m/s)": 8.3}
x_pos = np.arange(len(cats))
w = 0.35
for j, (deltas_arr, label, color) in enumerate([
    (deltas_oracle, "Oracle δ", "tab:blue"),
    (deltas_ls, "LS δ (20dB)", "tab:orange"),
]):
    vals = []
    for i, (cat_label, spd) in enumerate(cats.items()):
        mask = np.abs(speeds - spd) < 0.1
        vals.append(np.median(deltas_arr[mask]) if mask.any() else 0)
    ax.bar(x_pos + (j - 0.5) * w, vals, w, color=color, alpha=0.8, label=label)
ax.set_xticks(x_pos)
ax.set_xticklabels(cats.keys())
ax.set_ylabel("Median δ")
ax.set_title("δ by Mobility: Oracle vs LS")
ax.legend(fontsize=8)

# 1c: NMSE(reuse) by speed
ax = axes[2]
bar_colors = ["blue", "green", "red"]
for i, (label, spd) in enumerate({"static": 0.0, "ped (1m/s)": 1.0, "veh (8.3m/s)": 8.3}.items()):
    mask = np.abs(speeds - spd) < 0.1
    valid = nmses[mask]
    valid = valid[~np.isnan(valid)]
    if len(valid) > 0:
        med = np.median(valid)
        ax.bar(i, med, color=bar_colors[i], alpha=0.7, label=f"{label}: {med:.1f}dB")
ax.axhline(-10, color="gray", ls=":", alpha=0.5, label="-10dB threshold")
ax.set_xticks(range(3))
ax.set_xticklabels(["static", "ped (1m/s)", "veh (8.3m/s)"])
ax.set_ylabel("NMSE(reuse) [dB]")
ax.set_title("Quality Loss from Reuse")
ax.legend(fontsize=8)

fig.suptitle(f"S0: {PRESET} ({summary['n_ues']} UEs, {MAX_SNAPSHOTS} snaps)", fontsize=12)
plt.tight_layout()
plt.show()

## GO / NO-GO

In [4]:
check_go_nogo([result])

  ✓ GO: munich_elaa_m_1k_15g median δ = 0.007 ≤ 0.5


True